# Embeddings and Tokenization from Scratch

This notebook follows the core ideas from Chapter 2 of *Build a Large Language Model (From Scratch)* by Sebastian Raschka.

The goal is to understand how raw text is transformed into numerical representations (tokens and embeddings), and why this process is fundamental for Large Language Models and agentic AI systems.

In [1]:
import torch
import tiktoken

## Tokenization: Turning Text into Numbers

Large Language Models cannot operate directly on raw text. Neural networks work with numbers, so the first step is to convert text into numerical representations called *tokens*.

Modern LLMs use subword tokenization techniques (such as Byte Pair Encoding) to efficiently represent language. This allows models to handle rare words, new vocabulary, and misspellings without requiring an excessively large vocabulary.

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

len(text)

20479

In [4]:
tokens = tokenizer.encode(text)
len(tokens)

5145

## Context Windows and max_length

LLMs are trained to predict the next token given a fixed-size context window. The parameter max_length defines how many tokens the model can observe at once.

This is a crucial design choice:
- Short context windows may miss important dependencies in language.
- Longer context windows capture more information but increase computational cost.

Choosing an appropriate context length is a balance between model capability and efficiency.

In [5]:
def create_dataset(tokens, max_length, stride):
    input_ids = []
    target_ids = []

    for i in range(0, len(tokens) - max_length, stride):
        input_ids.append(tokens[i:i + max_length])
        target_ids.append(tokens[i + 1:i + max_length + 1])

    return torch.tensor(input_ids), torch.tensor(target_ids)

In [6]:
max_length = 32
stride = 16

X, y = create_dataset(tokens, max_length, stride)

X.shape, y.shape

(torch.Size([320, 32]), torch.Size([320, 32]))

## Overlapping Windows and Stride

The `stride` parameter controls how far the context window moves forward each time.

When `stride` is smaller than `max_length`, the windows overlap. This overlap is useful because it allows the model to see the same tokens in multiple contexts, improving learning efficiency and generalization.

Overlapping windows are especially important when training data is limited, as they effectively increase the number of training samples without requiring more text.

## Why Do Embeddings Encode Meaning?

Embeddings encode meaning because they are learned representations optimized through neural network training.

From a neural network perspective, embeddings are simply weight matrices that map token IDs to dense vectors. During training, tokens that appear in similar contexts receive similar gradient updates, causing their vectors to move closer together in embedding space.

As a result, semantic relationships emerge naturally: tokens with similar meanings are located near each other geometrically. This is why embeddings are so powerful for downstream tasks such as search, clustering, retrieval-augmented generation, and agentic reasoning.

## Experiment: Effect of `max_length` and `stride`

In this experiment, we vary the context window size (`max_length`) and the stride to observe how they affect the number of training samples.

This demonstrates the trade-off between context size, overlap, and dataset size.

In [7]:
settings = [
    (32, 16),
    (64, 32),
    (64, 8),
]

for max_length, stride in settings:
    X_tmp, y_tmp = create_dataset(tokens, max_length, stride)
    print(f"max_length={max_length}, stride={stride} → samples={X_tmp.shape[0]}")

max_length=32, stride=16 → samples=320
max_length=64, stride=32 → samples=159
max_length=64, stride=8 → samples=636


### Experiment Analysis

Increasing `max_length` results in fewer total samples because each training example consumes more tokens.

Reducing `stride` increases overlap between windows, which produces more samples. This overlap is beneficial because it exposes the model to more training signals and reinforces learning across different contexts.

This trade-off is a fundamental consideration when preparing datasets for training Large Language Models.